In [27]:
import dotenv
import sqlite3
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

dotenv.load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

In [28]:
class State(MessagesState):
    pass


graph_builder = StateGraph(State)

In [29]:
def chatbot(state: State):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


@tool
def get_weather(city: str):
    """Gets weather in city"""
    return f"The weather in {city} is sunny."


llm_with_tools = llm.bind_tools(tools=[get_weather])

In [30]:
tool_node = ToolNode(tools=[get_weather])

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(checkpointer=SqliteSaver(conn))

In [31]:
graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "What is the weather in machupichu?"},
        ],
    },
    config={"configurable": {"thread_id": 1}},
)

{'messages': [HumanMessage(content='What is the weather in machupichu?', additional_kwargs={}, response_metadata={}, id='acb07993-666d-4dd0-9e54-bd1c89147ce8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'id': 'chatcmpl-DbJUCfYHmRkT1X06bHmtpUz2CeZag', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dec44-e0fc-7b20-ab24-936b50e13933-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Machupichu'}, 'id': 'call_uiyXc661IIL02lFu9WhHaV7T', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5

In [32]:
graph.invoke(
    {
        "messages": [
            {"role": "user", "content": "Where did I asked about?"},
        ],
    },
    config={"configurable": {"thread_id": 1}},
)

{'messages': [HumanMessage(content='What is the weather in machupichu?', additional_kwargs={}, response_metadata={}, id='acb07993-666d-4dd0-9e54-bd1c89147ce8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 52, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_97c29773b5', 'id': 'chatcmpl-DbJUCfYHmRkT1X06bHmtpUz2CeZag', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dec44-e0fc-7b20-ab24-936b50e13933-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Machupichu'}, 'id': 'call_uiyXc661IIL02lFu9WhHaV7T', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5